In [6]:
import pandas as pd
import numpy as np

In [7]:
df = pd.read_csv('last_data_refined.csv')

In [25]:
df[df['TRANSACTION_DESCRIPTION'] == 'THOMPSON MASTERPIECE PIZZ']

,ACCOUNT_ID,CARD_ID,TRANSACTION_ID,GROSS_TRANSACTION_AMOUNT,TRANSACTION_DATE,TRANSACTION_TYPE,TRANSACTION_STATE,TRANSACTION_CITY,MERCHANT_STATE,MERCHANT_CITY,...,CARD_HOLDER_AVERAGE_LTM_TRANSACTION_COUNT,CARD_HOLDER_VINTAGE,CARD_PRESENT_INDICATOR,MERCHANT_ID,MERCHANT_NAME,SHOPPER_CLASSIFICATION,TZ_NAME,LOCAL_TIME,Hour,DayOfWeek
1453613,A3489876,C6937378,T8187082,42.85,2024-03-22 19:36:16+00:00,Spend,OH,THOMPSON,OH,NaN,...,46.33,87,Unknown,0,NO ENTITY,2,America/New_York,2024-03-22 15:36:16,15,Friday
1953442,A8712998,C8938244,T1452789,17.16,2024-02-03 22:03:40+00:00,Spend,OH,THOMPSON,OH,THOMPSON,...,91.83,66,Card Present,0,NO ENTITY,2,America/New_York,2024-02-03 17:03:40,17,Saturday


In [8]:
def apply_local_timezone(df):
    """
    TRANSACTION_STATE를 기반으로 UTC 시간을 각 지역의 정확한 현지 시간으로 변환하는 함수
    """
    print("시간대 변환을 시작합니다. (데이터 크기에 따라 수초~수십초 소요될 수 있습니다)")
    
    # 1. State별 주요 Timezone 매핑 딕셔너리
    # (일부 주에 2개의 시간대가 걸쳐있는 경우 가장 인구/상권이 많은 대표 시간대로 매핑했습니다)
    tz_mapping = {
        # Eastern Time (EST/EDT)
        'OH':'America/New_York', 'MI':'America/New_York', 'IN':'America/New_York', 'NC':'America/New_York', 
        'FL':'America/New_York', 'VA':'America/New_York', 'MA':'America/New_York', 'SC':'America/New_York', 
        'GA':'America/New_York', 'PA':'America/New_York', 'DE':'America/New_York', 'MD':'America/New_York', 
        'NY':'America/New_York', 'CT':'America/New_York', 'WV':'America/New_York', 'RI':'America/New_York', 
        'NJ':'America/New_York', 'DC':'America/New_York', 'ME':'America/New_York', 'VT':'America/New_York', 
        'NH':'America/New_York',
        
        # Central Time (CST/CDT)
        'MN':'America/Chicago', 'WI':'America/Chicago', 'TX':'America/Chicago', 'LA':'America/Chicago', 
        'IL':'America/Chicago', 'MS':'America/Chicago', 'TN':'America/Chicago', 'NE':'America/Chicago', 
        'MO':'America/Chicago', 'AL':'America/Chicago', 'ND':'America/Chicago', 'OK':'America/Chicago', 
        'IA':'America/Chicago', 'SD':'America/Chicago', 'AR':'America/Chicago', 'KS':'America/Chicago',
        
        # Mountain Time (MST/MDT)
        'CO':'America/Denver', 'UT':'America/Denver', 'NM':'America/Denver', 'ID':'America/Denver', 
        'MT':'America/Denver', 'WY':'America/Denver', 
        'AZ':'America/Phoenix', # 애리조나는 서머타임을 적용하지 않아 별도 분리
        
        # Pacific Time (PST/PDT)
        'CA':'America/Los_Angeles', 'NV':'America/Los_Angeles', 'WA':'America/Los_Angeles', 'OR':'America/Los_Angeles',
        
        # 기타 지역 및 해외 영토
        'AK':'America/Anchorage', 'HI':'Pacific/Honolulu', 
        'PR':'America/Puerto_Rico', 'VI':'America/Puerto_Rico', 'GU':'Pacific/Guam', 'MP':'Pacific/Guam', 'AS':'Pacific/Pago_Pago',
        
        # 군사 우편번호(AE, AP, AA) 등은 임의로 UTC나 동부시간으로 설정
        'AE':'UTC', 'AP':'UTC', 'AA':'America/New_York'
    }

    # 2. TRANSACTION_DATE를 Datetime 객체로 변환하고 기본값을 UTC로 명시
    if not pd.api.types.is_datetime64_any_dtype(df['TRANSACTION_DATE']):
        df['TRANSACTION_DATE'] = pd.to_datetime(df['TRANSACTION_DATE'])
    
    if df['TRANSACTION_DATE'].dt.tz is None:
        df['TRANSACTION_DATE'] = df['TRANSACTION_DATE'].dt.tz_localize('UTC')

    # 3. 주(State)에 맞는 타임존 열 생성 (결측치는 우선 UTC로 처리)
    df['TZ_NAME'] = df['TRANSACTION_STATE'].map(tz_mapping).fillna('America/New_York')

    # 4. 각 타임존별로 변환 결과를 담을 빈 시리즈 생성
    local_times = pd.Series(index=df.index, dtype='datetime64[ns]')

    # 5. 각 고유 타임존별로 현지 시간 변환 (서머타임 자동 계산됨)
    for tz in df['TZ_NAME'].unique():
        mask = df['TZ_NAME'] == tz
        # 해당 지역의 시간으로 변환 후, 다시 tz-naive 형태로 껍데기를 벗겨야 한 컬럼에 합칠 수 있음
        local_times[mask] = df.loc[mask, 'TRANSACTION_DATE'].dt.tz_convert(tz).dt.tz_localize(None)

    # 6. 최종 컬럼 적용 및 요일/시간 추출
    df['LOCAL_TIME'] = local_times
    df['Hour'] = df['LOCAL_TIME'].dt.hour
    df['DayOfWeek'] = df['LOCAL_TIME'].dt.day_name()
    
    # 요일 순서 정렬
    days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    df['DayOfWeek'] = pd.Categorical(df['DayOfWeek'], categories=days_order, ordered=True)

    print("✅ 현지 시간(Local Time) 변환이 완료되었습니다!")
    return df

# 함수 실행하여 df 업데이트
df = apply_local_timezone(df)

시간대 변환을 시작합니다. (데이터 크기에 따라 수초~수십초 소요될 수 있습니다)
✅ 현지 시간(Local Time) 변환이 완료되었습니다!


In [9]:
df['MERCHANT_CATEGORY_LEVEL_2'].value_counts()

MERCHANT_CATEGORY_LEVEL_2
QSR                                        2737443
Casual Dining                               932165
Vending & Beverage Retailers                550408
Food Services                               103970
General Retail                               91468
Payment Services                             85963
Grocery and Delivery                         20400
Food Delivery                                13586
Leisure                                      10069
Business Services                             7892
Specialty Retailers                           3951
Home Improvement and Furnishing               2256
Nutrition & Vitamin Retailers                 2117
Gas / Convenience                             2101
Department Stores                             1316
Travel                                        1222
Transportation                                 640
Apparel, Accessories, and Footwear             626
Lodging                                        318
Finan

### 주석 친 부분은 기본적인 정보 확인하는 코드

In [10]:
# import pandas as pd

# # 기본 정보 확인
# print("=== 기본 정보 ===")
# print(f"전체 행 수: {len(df):,}")
# print(f"컬럼 수: {df.shape[1]}")

# print("\n=== 핵심 컬럼 유니크 값 수 ===")
# cols = ['CARD_HOLDER_STATE', 'CARD_HOLDER_GENERATION', 
#         'MERCHANT_CATEGORY_LEVEL_2', 'MERCHANT_CATEGORY_LEVEL_3', 'MERCHANT_NAME']
# for col in cols:
#     print(f"{col}: {df[col].nunique()}개 (결측: {df[col].isna().sum():,}개)")

# print("\n=== CARD_HOLDER_STATE 분포 (상위 10개) ===")
# print(df['CARD_HOLDER_STATE'].value_counts().head(10))

# print("\n=== CARD_HOLDER_GENERATION 분포 ===")
# print(df['CARD_HOLDER_GENERATION'].value_counts())

# print("\n=== MERCHANT_CATEGORY_LEVEL_2 분포 (상위 15개) ===")
# print(df['MERCHANT_CATEGORY_LEVEL_2'].value_counts().head(15))

# print("\n=== STATE × GENERATION 조합별 거래 수 ===")
# combo = df.groupby(['CARD_HOLDER_STATE', 'CARD_HOLDER_GENERATION']).size()
# print(f"총 조합 수: {len(combo)}개")
# print(f"조합당 거래 수 - 평균: {combo.mean():.0f}, 최소: {combo.min()}, 최대: {combo.max()}")
# print(f"거래 수 100건 미만 조합: {(combo < 100).sum()}개")
# print(f"거래 수 1000건 이상 조합: {(combo >= 1000).sum()}개")

# print("\n=== basket당 아이템 수 미리보기 (STATE 기준) ===")
# basket_size = df.groupby('CARD_HOLDER_STATE')['MERCHANT_CATEGORY_LEVEL_2'].nunique()
# print(f"STATE 기준 basket당 MCL2 종류 수 - 평균: {basket_size.mean():.1f}, 최소: {basket_size.min()}, 최대: {basket_size.max()}")

# print("\n=== basket당 아이템 수 미리보기 (STATE × GENERATION 기준) ===")
# basket_size2 = df.groupby(['CARD_HOLDER_STATE', 'CARD_HOLDER_GENERATION'])['MERCHANT_CATEGORY_LEVEL_2'].nunique()
# print(f"STATE×GEN 기준 basket당 MCL2 종류 수 - 평균: {basket_size2.mean():.1f}, 최소: {basket_size2.min()}, 최대: {basket_size2.max()}")

In [11]:
# print("=== MCL3 분포 (상위 20개) ===")
# print(df['MERCHANT_CATEGORY_LEVEL_3'].value_counts().head(20))

# print("\n=== STATE×GEN basket당 MCL3 종류 수 ===")
# basket_mcl3 = df.groupby(['CARD_HOLDER_STATE', 'CARD_HOLDER_GENERATION'])['MERCHANT_CATEGORY_LEVEL_3'].nunique()
# print(f"평균: {basket_mcl3.mean():.1f}, 최소: {basket_mcl3.min()}, 최대: {basket_mcl3.max()}")

# print("\n=== MCL2 → MCL3 계층 구조 확인 ===")
# print(df.groupby(['MERCHANT_CATEGORY_LEVEL_2', 'MERCHANT_CATEGORY_LEVEL_3']).size().reset_index().groupby('MERCHANT_CATEGORY_LEVEL_2')['MERCHANT_CATEGORY_LEVEL_3'].count().sort_values(ascending=False).head(10))

In [12]:
# print("=== MERCHANT_NAME basket당 등장 수 (STATE×GEN 기준) ===")
# basket_name = df.groupby(['CARD_HOLDER_STATE', 'CARD_HOLDER_GENERATION'])['MERCHANT_NAME'].nunique()
# print(f"평균: {basket_name.mean():.1f}, 최소: {basket_name.min()}, 최대: {basket_name.max()}")

# print("\n=== MERCHANT_NAME 상위 30개 거래량 ===")
# print(df['MERCHANT_NAME'].value_counts().head(30))

# print("\n=== 식음료 관련 MCL2 필터링 후 MCL3 basket 밀도 ===")
# food_mcl2 = ['QSR', 'Casual Dining', 'Vending & Beverage Retailers',
#                 'Food Services', 'Grocery and Delivery', 'Food Delivery']
# df_food = df[df['MERCHANT_CATEGORY_LEVEL_2'].isin(food_mcl2)]
# print(f"필터링 후 행 수: {len(df_food):,} ({len(df_food)/len(df)*100:.1f}%)")

# basket_food = df_food.groupby(['CARD_HOLDER_STATE', 'CARD_HOLDER_GENERATION'])['MERCHANT_CATEGORY_LEVEL_3'].nunique()
# print(f"식음료 필터 후 basket당 MCL3 수 - 평균: {basket_food.mean():.1f}, 최소: {basket_food.min()}, 최대: {basket_food.max()}")

# basket_food_name = df_food.groupby(['CARD_HOLDER_STATE', 'CARD_HOLDER_GENERATION'])['MERCHANT_NAME'].nunique()
# print(f"식음료 필터 후 basket당 MERCHANT_NAME 수 - 평균: {basket_food_name.mean():.1f}, 최소: {basket_food_name.min()}, 최대: {basket_food_name.max()}")

In [13]:
# 결측치(NaN)를 제외하고 싶다면 df.dropna(subset=['merchant_category_level_2']) 를 먼저 적용할 수 있습니다.

# level_2 별로 level_3의 고유값들을 배열로 묶기
category_mapping = df.groupby('MERCHANT_CATEGORY_LEVEL_2')['MERCHANT_CATEGORY_LEVEL_3'].unique()

# 보기 좋게 출력하기
for level_2, level_3_list in category_mapping.items():
    print(f"▶ {level_2}")
    for level_3 in level_3_list:
        print(f"  - {level_3}")
    print("-" * 40)

▶ Apparel, Accessories, and Footwear
  - Women's Apparel
  - Luxury Retailers
  - Teen & Young Adult Apparel
  - Apparel, Accessories, and Footwear
  - Athletic Apparel & Footwear
  - Family Apparel
  - Footwear
  - Luxury Goods
  - Outdoor Apparel
----------------------------------------
▶ Auto Insurance
  - Auto Insurance
----------------------------------------
▶ Automobiles, Parts, Tires, and Services
  - Automobile Parts, Tires, and Services
  - Automobile Dealership
  - Car Wash
----------------------------------------
▶ Business Services
  - Business Services
----------------------------------------
▶ Casual Dining
  - American Restaurants
  - Breakfast Restaurants
  - Pizza Restaurants
  - Mexican Restaurants
  - Asian Restaurants
  - Seafood Restaurants
  - Carribean
  - Italian Restaurants
  - Mediterranean Restaurants
----------------------------------------
▶ Commercial Equipment
  - Commercial Equipment
----------------------------------------
▶ Consumer Services
  - Barbe

### CARD_HOLDER_STATEM CARD_HOLDER_GENERATION을 집단으로 보기
### 카테고리는 음식 관련으로 필터링 해주기!

In [14]:
# ================================
# 1. 전처리
# ================================

# 식음료 필터링
food_mcl2 = ['QSR', 'Casual Dining', 'Vending & Beverage Retailers', 'Food Services', 'Grocery and Delivery', 'Food Delivery',
            'Gas / Convenience', 'Nutrition & Vitamin Retailers']

df_food = df[
    df['MERCHANT_CATEGORY_LEVEL_2'].isin(food_mcl2) &
    df['CARD_HOLDER_STATE'].notna() &
    df['CARD_HOLDER_GENERATION'].notna() &
    df['MERCHANT_CATEGORY_LEVEL_3'].notna()
].copy()

# 거래 수 100건 미만 STATE×GEN 조합 제거
combo_counts = df_food.groupby(['CARD_HOLDER_STATE', 'CARD_HOLDER_GENERATION']).size()
valid_combos = combo_counts[combo_counts >= 100].index
df_food = df_food.set_index(['CARD_HOLDER_STATE', 'CARD_HOLDER_GENERATION'])
df_food = df_food[df_food.index.isin(valid_combos)].reset_index()

print(f"전처리 후 행 수: {len(df_food):,}")
print(f"유효 STATE×GEN 조합 수: {df_food.groupby(['CARD_HOLDER_STATE','CARD_HOLDER_GENERATION']).ngroups}")
print(f"MCL3 종류 수: {df_food['MERCHANT_CATEGORY_LEVEL_3'].nunique()}")

# ================================
# 2. 전체 평균 비율 (기댓값)
# ================================

# 전체에서 MCL3별 거래 비율
global_ratio = (
    df_food['MERCHANT_CATEGORY_LEVEL_3']
    .value_counts(normalize=True)
    .rename('global_ratio')
)

print("\n=== 전체 MCL3 비율 ===")
print((global_ratio * 100).round(2).head(15))

# ================================
# 3. 집단별 MCL3 비율 계산
# ================================

group_counts = (
    df_food.groupby(['CARD_HOLDER_STATE', 'CARD_HOLDER_GENERATION', 'MERCHANT_CATEGORY_LEVEL_3'])
    .size()
    .reset_index(name='count')
)

# 집단별 전체 거래 수
group_total = (
    df_food.groupby(['CARD_HOLDER_STATE', 'CARD_HOLDER_GENERATION'])
    .size()
    .reset_index(name='total')
)
group_ratio = group_counts.merge(group_total, on=['CARD_HOLDER_STATE', 'CARD_HOLDER_GENERATION'])
group_ratio['local_ratio'] = group_ratio['count'] / group_ratio['total']

# ================================
# 4. Lift 계산
# ================================

group_ratio = group_ratio.merge(
    global_ratio.reset_index().rename(columns={'MERCHANT_CATEGORY_LEVEL_3': 'MERCHANT_CATEGORY_LEVEL_3'}),
    on='MERCHANT_CATEGORY_LEVEL_3'
)
group_ratio['lift'] = group_ratio['local_ratio'] / group_ratio['global_ratio']

# ================================
# 5. 결과 확인
# ================================

print("\n=== lift 상위 30개 (전국 평균 대비 특이 소비 패턴) ===")
top_lift = (
    group_ratio[group_ratio['count'] >= 50]  # 최소 거래 수 필터
    .sort_values('lift', ascending=False)
    [['CARD_HOLDER_STATE', 'CARD_HOLDER_GENERATION', 'MERCHANT_CATEGORY_LEVEL_3',
      'count', 'local_ratio', 'global_ratio', 'lift']]
    .head(30)
)
top_lift['local_ratio'] = (top_lift['local_ratio'] * 100).round(2)
top_lift['global_ratio'] = (top_lift['global_ratio'] * 100).round(2)
top_lift['lift'] = top_lift['lift'].round(3)
print(top_lift.to_string(index=False))

print("\n=== 세대별 특이 MCL3 Top 5 (lift 기준) ===")
for gen in df_food['CARD_HOLDER_GENERATION'].unique():
    print(f"\n[{gen}]")
    gen_data = group_ratio[
        (group_ratio['CARD_HOLDER_GENERATION'] == gen) &
        (group_ratio['count'] >= 50)
    ].sort_values('lift', ascending=False)
    print(gen_data[['CARD_HOLDER_STATE', 'MERCHANT_CATEGORY_LEVEL_3', 'lift', 'count']]
          .head(15).to_string(index=False))

print("\n=== MCL3별 세대 간 lift 차이 (세대 구분력이 높은 카테고리) ===")
gen_pivot = (
    group_ratio[group_ratio['count'] >= 50]
    .groupby(['CARD_HOLDER_GENERATION', 'MERCHANT_CATEGORY_LEVEL_3'])['lift']
    .mean()
    .unstack('CARD_HOLDER_GENERATION')
    .fillna(0)
)
gen_pivot['lift_range'] = gen_pivot.max(axis=1) - gen_pivot.min(axis=1)
print(gen_pivot.sort_values('lift_range', ascending=False).head(15).round(3))

전처리 후 행 수: 4,308,869
유효 STATE×GEN 조합 수: 251
MCL3 종류 수: 22

=== 전체 MCL3 비율 ===
MERCHANT_CATEGORY_LEVEL_3
QSR Burgers                     30.62
Vending & Beverage Retailers    12.65
Coffee / Tea                    11.47
QSR Chicken                      9.92
Mexican Restaurants              8.04
QSR Sandwiches                   6.18
American Restaurants             4.51
Breakfast Restaurants            4.30
Pizza Restaurants                3.99
Dessert                          2.44
Food Services                    2.39
Asian Restaurants                1.49
QSR Smoothies                    0.74
Grocery                          0.47
Food Delivery                    0.31
Name: global_ratio, dtype: float64

=== lift 상위 30개 (전국 평균 대비 특이 소비 패턴) ===
CARD_HOLDER_STATE CARD_HOLDER_GENERATION     MERCHANT_CATEGORY_LEVEL_3  count  local_ratio  global_ratio   lift
               VT                  Gen X Nutrition & Vitamin Retailers     57         1.26          0.05 25.950
               VT         

In [15]:
# ================================
# 세대 효과 vs 지역 효과 분리
# ================================

# 세대별 전국 평균 lift (지역 무관)
print("=== 세대별 MCL3 전국 평균 lift ===")
gen_total = df_food.groupby('CARD_HOLDER_GENERATION').size().reset_index(name='gen_total')
gen_counts = (
    df_food.groupby(['CARD_HOLDER_GENERATION', 'MERCHANT_CATEGORY_LEVEL_3'])
    .size().reset_index(name='count')
    .merge(gen_total, on='CARD_HOLDER_GENERATION')
)
gen_counts['local_ratio'] = gen_counts['count'] / gen_counts['gen_total']
gen_counts = gen_counts.merge(
    global_ratio.reset_index(),
    on='MERCHANT_CATEGORY_LEVEL_3'
)
gen_counts['lift'] = gen_counts['local_ratio'] / gen_counts['global_ratio']

gen_pivot_clean = (
    gen_counts.pivot(index='MERCHANT_CATEGORY_LEVEL_3', 
                     columns='CARD_HOLDER_GENERATION', 
                     values='lift')
    .round(3)
)
gen_pivot_clean['lift_range'] = gen_pivot_clean.max(axis=1) - gen_pivot_clean.min(axis=1)
print(gen_pivot_clean.sort_values('lift_range', ascending=False).to_string())

# 주별 전국 평균 lift (세대 무관)
print("\n=== 주별 MCL3 lift (세대 무관, lift 상위 패턴만) ===")
state_total = df_food.groupby('CARD_HOLDER_STATE').size().reset_index(name='state_total')
state_counts = (
    df_food.groupby(['CARD_HOLDER_STATE', 'MERCHANT_CATEGORY_LEVEL_3'])
    .size().reset_index(name='count')
    .merge(state_total, on='CARD_HOLDER_STATE')
)
state_counts['local_ratio'] = state_counts['count'] / state_counts['state_total']
state_counts = state_counts.merge(
    global_ratio.reset_index(),
    on='MERCHANT_CATEGORY_LEVEL_3'
)
state_counts['lift'] = state_counts['local_ratio'] / state_counts['global_ratio']

# 주별로 lift 가장 높은 MCL3 하나씩
state_top = (
    state_counts[state_counts['count'] >= 50]
    .sort_values('lift', ascending=False)
    .groupby('CARD_HOLDER_STATE')
    .first()
    .reset_index()
    [['CARD_HOLDER_STATE', 'MERCHANT_CATEGORY_LEVEL_3', 'lift', 'count']]
    .sort_values('lift', ascending=False)
    .head(20)
)
state_top['lift'] = state_top['lift'].round(3)
print(state_top.to_string(index=False))

# ================================
# 세대 × 지역 교차 효과 (순수 세대 효과 제거 후)
# ================================
print("\n=== 순수 지역×세대 교차 효과 (전국 세대 평균 대비) ===")
# 집단 lift / 세대 전국 lift → 1보다 크면 "세대 평균보다도 이 지역이 더 특이"
group_ratio2 = group_ratio.merge(
    gen_counts[['CARD_HOLDER_GENERATION', 'MERCHANT_CATEGORY_LEVEL_3', 'lift']]
    .rename(columns={'lift': 'gen_lift'}),
    on=['CARD_HOLDER_GENERATION', 'MERCHANT_CATEGORY_LEVEL_3']
)
group_ratio2['cross_lift'] = group_ratio2['lift'] / group_ratio2['gen_lift']

print(
    group_ratio2[group_ratio2['count'] >= 100]
    .sort_values('cross_lift', ascending=False)
    [['CARD_HOLDER_STATE', 'CARD_HOLDER_GENERATION', 'MERCHANT_CATEGORY_LEVEL_3',
        'count', 'lift', 'gen_lift', 'cross_lift']]
    .head(20)
    .round(3)
    .to_string(index=False)
)

=== 세대별 MCL3 전국 평균 lift ===
CARD_HOLDER_GENERATION         Baby Boomer  Gen X  Gen Z  Millennial  Silent  lift_range
MERCHANT_CATEGORY_LEVEL_3                                                               
Seafood Restaurants                  1.984  1.059  0.643       0.726   3.926       3.283
Carribean                            1.404  1.274  0.677       0.785   3.090       2.413
Italian Restaurants                  1.460  1.261  0.724       0.765   1.976       1.252
Food Delivery                        0.675  0.806  1.567       1.011   0.343       1.224
American Restaurants                 1.373  1.112  0.820       0.857   1.867       1.047
Mediterranean Restaurants              NaN  1.640  0.603       1.113     NaN       1.037
Vending & Beverage Retailers         0.611  0.825  1.280       1.147   0.245       1.035
QSR Sandwiches                       1.368  1.116  0.737       0.894   1.684       0.947
Nutrition & Vitamin Retailers        0.852  1.167  0.874       1.002   0.345      

In [16]:
gr = group_ratio2[group_ratio2['count'] >= 100].sort_values('cross_lift', ascending=False)[['CARD_HOLDER_STATE', 'CARD_HOLDER_GENERATION', 'MERCHANT_CATEGORY_LEVEL_3','count', 'lift', 'gen_lift', 'cross_lift']].head(50).round(3)

In [17]:
gr.head(30)

,CARD_HOLDER_STATE,CARD_HOLDER_GENERATION,MERCHANT_CATEGORY_LEVEL_3,count,lift,gen_lift,cross_lift
882,FL,Millennial,Carribean,618,11.346,0.785,14.448
840,FL,Gen X,Carribean,648,16.109,1.274,12.641
819,FL,Baby Boomer,Carribean,392,17.498,1.404,12.461
862,FL,Gen Z,Carribean,229,7.172,0.677,10.598
339,AZ,Gen X,QSR Healthy,367,10.479,1.000,10.479
988,GA,Millennial,Food Delivery,1320,10.285,1.011,10.175
1751,LA,Millennial,Food Delivery,468,10.182,1.011,10.073
967,GA,Gen Z,Food Delivery,1077,15.691,1.567,10.011
379,AZ,Millennial,QSR Healthy,500,9.491,1.040,9.124
946,GA,Gen X,Food Delivery,577,7.282,0.806,9.040


In [18]:
df[(df['MERCHANT_CATEGORY_LEVEL_3'] == 'Carribean') & (df['CARD_HOLDER_STATE'] == 'PR')][['MERCHANT_CATEGORY_LEVEL_2', 'MERCHANT_CATEGORY_LEVEL_3', 'MERCHANT_NAME', 'CARD_HOLDER_STATE']].drop_duplicates()

,MERCHANT_CATEGORY_LEVEL_2,MERCHANT_CATEGORY_LEVEL_3,MERCHANT_NAME,CARD_HOLDER_STATE
95636,Casual Dining,Carribean,POLLO TROPICAL,PR


In [19]:
condition = (df['MERCHANT_CATEGORY_LEVEL_3'] == 'QSR Healthy') & (df['CARD_HOLDER_STATE'] == 'AZ')
df[condition].groupby(
    ['MERCHANT_CATEGORY_LEVEL_2', 'MERCHANT_CATEGORY_LEVEL_3', 'MERCHANT_NAME', 'CARD_HOLDER_STATE'], 
    as_index=False
)['GROSS_TRANSACTION_AMOUNT'].sum()

,MERCHANT_CATEGORY_LEVEL_2,MERCHANT_CATEGORY_LEVEL_3,MERCHANT_NAME,CARD_HOLDER_STATE,GROSS_TRANSACTION_AMOUNT
0,QSR,QSR Healthy,NO ENTITY,AZ,1034.29
1,QSR,QSR Healthy,SALAD AND GO,AZ,12071.23
2,QSR,QSR Healthy,SWEETGREEN,AZ,159.95


In [20]:
gen_pivot_clean.sort_values('lift_range', ascending=False)

CARD_HOLDER_GENERATION,Baby Boomer,Gen X,Gen Z,Millennial,Silent,lift_range
MERCHANT_CATEGORY_LEVEL_3,,,,,,
Seafood Restaurants,1.984,1.059,0.643,0.726,3.926,3.283
Carribean,1.404,1.274,0.677,0.785,3.090,2.413
Italian Restaurants,1.460,1.261,0.724,0.765,1.976,1.252
Food Delivery,0.675,0.806,1.567,1.011,0.343,1.224
American Restaurants,1.373,1.112,0.820,0.857,1.867,1.047
Mediterranean Restaurants,NaN,1.640,0.603,1.113,NaN,1.037
Vending & Beverage Retailers,0.611,0.825,1.280,1.147,0.245,1.035
QSR Sandwiches,1.368,1.116,0.737,0.894,1.684,0.947
Nutrition & Vitamin Retailers,0.852,1.167,0.874,1.002,0.345,0.822


In [21]:
import pandas as pd
import numpy as np

# 지역 매핑
region_map = {
    'CT': 'Northeast', 'ME': 'Northeast', 'MA': 'Northeast', 'NH': 'Northeast', 
    'RI': 'Northeast', 'VT': 'Northeast', 'NJ': 'Northeast', 'NY': 'Northeast', 'PA': 'Northeast',
    'IL': 'Midwest', 'IN': 'Midwest', 'MI': 'Midwest', 'OH': 'Midwest', 'WI': 'Midwest', 
    'IA': 'Midwest', 'KS': 'Midwest', 'MN': 'Midwest', 'MO': 'Midwest', 'NE': 'Midwest', 
    'ND': 'Midwest', 'SD': 'Midwest',
    'DE': 'South', 'FL': 'South', 'GA': 'South', 'MD': 'South', 'NC': 'South', 
    'SC': 'South', 'VA': 'South', 'DC': 'South', 'WV': 'South', 'AL': 'South', 
    'KY': 'South', 'MS': 'South', 'TN': 'South', 'AR': 'South', 'LA': 'South', 
    'OK': 'South', 'TX': 'South',
    'AZ': 'West', 'CO': 'West', 'ID': 'West', 'MT': 'West', 'NV': 'West', 
    'NM': 'West', 'UT': 'West', 'WY': 'West', 'AK': 'West', 'CA': 'West', 
    'HI': 'West', 'OR': 'West', 'WA': 'West'
}

df_food['REGION'] = df_food['CARD_HOLDER_STATE'].map(region_map)

# 매핑 안된 주 확인
unmapped = df_food[df_food['REGION'].isna()]['CARD_HOLDER_STATE'].unique()
print(f"매핑 안된 주: {unmapped}")

# ================================
# 1. 지역별 전체 lift
# ================================
print("\n=== 지역별 MCL3 Lift (전국 평균 대비) ===")

region_total = df_food.groupby('REGION').size().reset_index(name='region_total')
region_counts = (
    df_food.groupby(['REGION', 'MERCHANT_CATEGORY_LEVEL_3'])
    .size().reset_index(name='count')
    .merge(region_total, on='REGION')
)
region_counts['local_ratio'] = region_counts['count'] / region_counts['region_total']
region_counts = region_counts.merge(
    global_ratio.reset_index(), on='MERCHANT_CATEGORY_LEVEL_3'
)
region_counts['lift'] = region_counts['local_ratio'] / region_counts['global_ratio']

region_pivot = (
    region_counts.pivot(index='MERCHANT_CATEGORY_LEVEL_3', columns='REGION', values='lift')
    .round(3)
)
region_pivot['lift_range'] = region_pivot.max(axis=1) - region_pivot.min(axis=1)
print(region_pivot.sort_values('lift_range', ascending=False).to_string())

# ================================
# 2. 남부 × 세대 집중 분석
# ================================
print("\n=== 남부(South) 주별 × 세대별 MCL3 Lift (lift >= 2.0, count >= 50) ===")

df_south = df_food[df_food['REGION'] == 'South']

south_total = df_south.groupby(['CARD_HOLDER_STATE', 'CARD_HOLDER_GENERATION']).size().reset_index(name='total')
south_counts = (
    df_south.groupby(['CARD_HOLDER_STATE', 'CARD_HOLDER_GENERATION', 'MERCHANT_CATEGORY_LEVEL_3'])
    .size().reset_index(name='count')
    .merge(south_total, on=['CARD_HOLDER_STATE', 'CARD_HOLDER_GENERATION'])
)
south_counts['local_ratio'] = south_counts['count'] / south_counts['total']
south_counts = south_counts.merge(global_ratio.reset_index(), on='MERCHANT_CATEGORY_LEVEL_3')
south_counts['lift'] = south_counts['local_ratio'] / south_counts['global_ratio']

south_filtered = south_counts[
    (south_counts['lift'] >= 2.0) & 
    (south_counts['count'] >= 50)
].sort_values('lift', ascending=False)

print(south_filtered[['CARD_HOLDER_STATE', 'CARD_HOLDER_GENERATION', 
                        'MERCHANT_CATEGORY_LEVEL_3', 'count', 'lift']]
      .to_string(index=False))

# ================================
# 3. 남부 내 세대별 특화 MCL3
# ================================
print("\n=== 남부 한정 세대별 lift 피벗 ===")

south_gen = (
    df_south.groupby(['CARD_HOLDER_GENERATION', 'MERCHANT_CATEGORY_LEVEL_3'])
    .size().reset_index(name='count')
)
south_gen_total = df_south.groupby('CARD_HOLDER_GENERATION').size().reset_index(name='gen_total')
south_gen = south_gen.merge(south_gen_total, on='CARD_HOLDER_GENERATION')
south_gen['local_ratio'] = south_gen['count'] / south_gen['gen_total']
south_gen = south_gen.merge(global_ratio.reset_index(), on='MERCHANT_CATEGORY_LEVEL_3')
south_gen['lift'] = south_gen['local_ratio'] / south_gen['global_ratio']

south_gen_pivot = (
    south_gen.pivot(index='MERCHANT_CATEGORY_LEVEL_3', 
                    columns='CARD_HOLDER_GENERATION', values='lift')
    .round(3)
)
south_gen_pivot['lift_range'] = south_gen_pivot.max(axis=1) - south_gen_pivot.min(axis=1)
print(south_gen_pivot.sort_values('lift_range', ascending=False).to_string())

# ================================
# 4. 남부 Food Delivery 세대별 주별 집중도
# ================================
print("\n=== 남부 Food Delivery 세대×주 히트맵 ===")

south_delivery = south_counts[
    (south_counts['MERCHANT_CATEGORY_LEVEL_3'] == 'Food Delivery') &
    (south_counts['count'] >= 30)
].pivot(index='CARD_HOLDER_STATE', columns='CARD_HOLDER_GENERATION', values='lift').round(2)
south_delivery
print(south_delivery.to_string())

매핑 안된 주: ['PR' 'GU' 'AE' 'AP']

=== 지역별 MCL3 Lift (전국 평균 대비) ===
REGION                         Midwest  Northeast  South   West  lift_range
MERCHANT_CATEGORY_LEVEL_3                                                  
Breakfast Restaurants            1.020      3.893  0.769  0.219       3.674
Carribean                        0.031      0.078  2.220  0.035       2.189
Food Delivery                    0.479      0.285  1.893  0.116       1.777
QSR Healthy                      0.413      2.047  0.869  1.920       1.634
Italian Restaurants              1.699      0.192  0.784  0.614       1.507
Asian Restaurants                0.737      0.497  0.904  1.929       1.432
Seafood Restaurants              0.477      0.424  1.763  0.386       1.377
Nutrition & Vitamin Retailers    0.954      1.453  1.246  0.293       1.160
QSR Smoothies                    0.931      0.493  1.409  0.378       1.031
QSR Chicken                      0.548      0.569  1.536  0.729       0.988
Mediterranean Restauran

In [22]:
df[df['MERCHANT_CATEGORY_LEVEL_2'] == 'Food Delivery'][['MERCHANT_CATEGORY_LEVEL_2', 'MERCHANT_NAME']].drop_duplicates()

,MERCHANT_CATEGORY_LEVEL_2,MERCHANT_NAME
112,Food Delivery,FIVE STAR FOOD SERVICE
9235,Food Delivery,SCHWANS
13413,Food Delivery,NO ENTITY
367030,Food Delivery,DOORDASH DASHPASS


In [23]:
region_map = {
    'CT': 'Northeast', 'ME': 'Northeast', 'MA': 'Northeast', 'NH': 'Northeast', 
    'RI': 'Northeast', 'VT': 'Northeast', 'NJ': 'Northeast', 'NY': 'Northeast', 'PA': 'Northeast',
    'IL': 'Midwest', 'IN': 'Midwest', 'MI': 'Midwest', 'OH': 'Midwest', 'WI': 'Midwest', 
    'IA': 'Midwest', 'KS': 'Midwest', 'MN': 'Midwest', 'MO': 'Midwest', 'NE': 'Midwest', 
    'ND': 'Midwest', 'SD': 'Midwest',
    'DE': 'South', 'FL': 'South', 'GA': 'South', 'MD': 'South', 'NC': 'South', 
    'SC': 'South', 'VA': 'South', 'DC': 'South', 'WV': 'South', 'AL': 'South', 
    'KY': 'South', 'MS': 'South', 'TN': 'South', 'AR': 'South', 'LA': 'South', 
    'OK': 'South', 'TX': 'South',
    'AZ': 'West', 'CO': 'West', 'ID': 'West', 'MT': 'West', 'NV': 'West', 
    'NM': 'West', 'UT': 'West', 'WY': 'West', 'AK': 'West', 'CA': 'West', 
    'HI': 'West', 'OR': 'West', 'WA': 'West'
}
df_food['REGION'] = df_food['CARD_HOLDER_STATE'].map(region_map)

# ================================
# 1. TX·FL Food Delivery 실태 확인
#    (NaN = count<30 필터 때문인지, 진짜 거래가 없는지)
# ================================
print("=== TX·FL Food Delivery 원본 거래 수 ===")
check = df_food[
    (df_food['CARD_HOLDER_STATE'].isin(['TX', 'FL', 'GA', 'LA'])) &
    (df_food['MERCHANT_CATEGORY_LEVEL_3'] == 'Food Delivery')
].groupby(['CARD_HOLDER_STATE', 'CARD_HOLDER_GENERATION']).size().reset_index(name='count')
print(check.to_string(index=False))

print("\n=== TX·FL 전체 거래 수 (분모 확인) ===")
total_check = df_food[
    df_food['CARD_HOLDER_STATE'].isin(['TX', 'FL', 'GA', 'LA'])
].groupby(['CARD_HOLDER_STATE', 'CARD_HOLDER_GENERATION']).size().reset_index(name='total')
print(total_check.to_string(index=False))

# ================================
# 2. 요일별 거래 패턴 — 남부 vs 타 지역
# ================================
print("\n=== 지역별 요일별 거래 비율 ===")
df_food['DayOfWeek'] = pd.Categorical(
    df_food['DayOfWeek'],
    categories=['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'],
    ordered=True
)

day_region = (
    df_food[df_food['REGION'].notna()]
    .groupby(['REGION', 'DayOfWeek'])
    .size().reset_index(name='count')
)
day_region_total = df_food[df_food['REGION'].notna()].groupby('REGION').size().reset_index(name='total')
day_region = day_region.merge(day_region_total, on='REGION')
day_region['ratio'] = day_region['count'] / day_region['total'] * 100

day_pivot = day_region.pivot(index='DayOfWeek', columns='REGION', values='ratio').round(2)
print(day_pivot.to_string())

# ================================
# 3. 일요일 MCL3 구성 — 남부 vs 타 지역
# ================================
print("\n=== 일요일 한정 지역별 MCL3 lift (vs 해당 지역 전체 평균) ===")
df_sunday = df_food[
    (df_food['DayOfWeek'] == 'Sunday') & 
    (df_food['REGION'].notna())
]

# 일요일 지역별 MCL3 비율
sun_counts = df_sunday.groupby(['REGION', 'MERCHANT_CATEGORY_LEVEL_3']).size().reset_index(name='sun_count')
sun_total = df_sunday.groupby('REGION').size().reset_index(name='sun_total')
sun_ratio = sun_counts.merge(sun_total, on='REGION')
sun_ratio['sun_ratio'] = sun_ratio['sun_count'] / sun_ratio['sun_total']

# 전체 지역별 MCL3 비율 (평일 포함)
all_counts = df_food[df_food['REGION'].notna()].groupby(['REGION', 'MERCHANT_CATEGORY_LEVEL_3']).size().reset_index(name='all_count')
all_total = df_food[df_food['REGION'].notna()].groupby('REGION').size().reset_index(name='all_total')
all_ratio = all_counts.merge(all_total, on='REGION')
all_ratio['all_ratio'] = all_ratio['all_count'] / all_ratio['all_total']

sun_lift = sun_ratio.merge(all_ratio[['REGION','MERCHANT_CATEGORY_LEVEL_3','all_ratio']], 
                            on=['REGION','MERCHANT_CATEGORY_LEVEL_3'])
sun_lift['sunday_lift'] = sun_lift['sun_ratio'] / sun_lift['all_ratio']

sun_pivot = sun_lift.pivot(index='MERCHANT_CATEGORY_LEVEL_3', columns='REGION', values='sunday_lift').round(3)
sun_pivot['South_vs_others'] = sun_pivot['South'] - sun_pivot[['Midwest','Northeast','West']].mean(axis=1)
print(sun_pivot.sort_values('South_vs_others', ascending=False).to_string())

# ================================
# 4. 남부 요일별 × MCL3 패턴
# ================================
print("\n=== 남부 한정 요일별 MCL3 lift (vs 남부 전체 평균) ===")
df_south_day = df_food[df_food['REGION'] == 'South']

south_day_counts = df_south_day.groupby(['DayOfWeek', 'MERCHANT_CATEGORY_LEVEL_3']).size().reset_index(name='count')
south_day_total = df_south_day.groupby('DayOfWeek').size().reset_index(name='total')
south_day_ratio = south_day_counts.merge(south_day_total, on='DayOfWeek')
south_day_ratio['day_ratio'] = south_day_ratio['count'] / south_day_ratio['total']

south_avg = df_south_day.groupby('MERCHANT_CATEGORY_LEVEL_3').size()
south_avg = (south_avg / len(df_south_day)).reset_index(name='avg_ratio')
south_avg.columns = ['MERCHANT_CATEGORY_LEVEL_3', 'avg_ratio']

south_day_lift = south_day_ratio.merge(south_avg, on='MERCHANT_CATEGORY_LEVEL_3')
south_day_lift['lift'] = south_day_lift['day_ratio'] / south_day_lift['avg_ratio']

south_day_pivot = south_day_lift.pivot(
    index='MERCHANT_CATEGORY_LEVEL_3', columns='DayOfWeek', values='lift'
).round(3)
print(south_day_pivot.to_string())

=== TX·FL Food Delivery 원본 거래 수 ===
CARD_HOLDER_STATE CARD_HOLDER_GENERATION  count
               FL            Baby Boomer     18
               FL                  Gen X     43
               FL                  Gen Z     54
               FL             Millennial     67
               GA            Baby Boomer    273
               GA                  Gen X    577
               GA                  Gen Z   1077
               GA             Millennial   1320
               GA                 Silent      8
               LA            Baby Boomer    104
               LA                  Gen X    266
               LA                  Gen Z    333
               LA             Millennial    468
               LA                 Silent      4
               TX            Baby Boomer     13
               TX                  Gen X     52
               TX                  Gen Z     32
               TX             Millennial     83

=== TX·FL 전체 거래 수 (분모 확인) ===
CARD_HOLDER_STATE CAR

C:\Users\yunji\AppData\Local\Temp\ipykernel_13788\3894701398.py:46: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(['REGION', 'DayOfWeek'])


REGION     Midwest  Northeast  South   West
DayOfWeek                                  
Monday       13.14      12.81  13.27  13.32
Tuesday      14.25      13.97  14.05  14.13
Wednesday    15.16      14.88  14.95  14.78
Thursday     16.21      15.79  15.99  15.55
Friday       17.06      17.01  16.99  16.56
Saturday     13.52      14.26  14.18  14.23
Sunday       10.66      11.28  10.58  11.43

=== 일요일 한정 지역별 MCL3 lift (vs 해당 지역 전체 평균) ===
REGION                         Midwest  Northeast  South   West  South_vs_others
MERCHANT_CATEGORY_LEVEL_3                                                       
Dessert                          1.039      1.001  1.319  1.239         0.226000
Pizza Restaurants                1.329      1.259  1.355  1.205         0.090667
QSR Healthy                      0.775      0.636  0.818  0.830         0.071000
QSR Burgers                      1.103      1.101  1.158  1.076         0.064667
Mexican Restaurants              1.098      1.024  1.098  1.011        

C:\Users\yunji\AppData\Local\Temp\ipykernel_13788\3894701398.py:91: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  south_day_counts = df_south_day.groupby(['DayOfWeek', 'MERCHANT_CATEGORY_LEVEL_3']).size().reset_index(name='count')
C:\Users\yunji\AppData\Local\Temp\ipykernel_13788\3894701398.py:92: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  south_day_total = df_south_day.groupby('DayOfWeek').size().reset_index(name='total')


DayOfWeek                      Monday  Tuesday  Wednesday  Thursday  Friday  Saturday  Sunday
MERCHANT_CATEGORY_LEVEL_3                                                                    
American Restaurants            0.953    0.906      0.918     0.944   0.992     1.111   1.249
Asian Restaurants               0.945    0.957      0.994     0.998   1.025     0.997   1.101
Breakfast Restaurants           0.986    0.949      0.966     0.923   0.998     1.046   1.191
Carribean                       1.048    1.037      0.985     1.028   0.962     0.881   1.090
Coffee / Tea                    0.990    0.996      0.968     0.981   0.970     1.038   1.089
Dessert                         0.939    0.881      0.875     0.875   0.962     1.255   1.319
Food Delivery                   0.977    1.134      1.147     1.148   0.996     0.860   0.614
Food Services                   0.982    1.063      1.022     1.055   1.004     0.970   0.858
Gas / Convenience               1.150    0.887      1.075   